# Solar Interior Rotation and its Variation — Implementation\n# 태양 내부 자전과 그 변동 — 구현\n\n**Paper**: Howe, R. (2009), *Living Rev. Solar Phys.*, **6**, 1\n\nThis notebook implements key concepts from the review:\n이 노트북은 리뷰의 핵심 개념을 구현합니다:\n\n1. **Solar rotation profile visualization** / 태양 자전 프로파일 시각화\n2. **Rotational splitting from differential rotation** / 차등 자전에 의한 자전 분리\n3. **Simplified 1D rotation inversion (RLS)** / 간소화된 1D 자전 역산 (RLS)\n4. **Torsional oscillation pattern** / 비틀림 진동 패턴\n5. **Near-surface shear layer** / 표면 근처 전단층

In [ ]:
import numpy as np\nimport matplotlib.pyplot as plt\nfrom matplotlib import cm\nfrom matplotlib.colors import Normalize\nfrom scipy.special import legendre\nfrom scipy.optimize import minimize\nimport matplotlib.gridspec as gridspec\n\nplt.rcParams.update({\n    \"figure.dpi\": 120,\n    \"font.size\": 11,\n    \"axes.titlesize\": 13,\n    \"figure.facecolor\": \"white\",\n})

## 1. Solar Rotation Profile Model / 태양 자전 프로파일 모델\n\nWe build a realistic model of the solar internal rotation based on the observational picture described in the review (Figure 1). The model captures:\n리뷰(Figure 1)에 기술된 관측적 그림을 바탕으로 태양 내부 자전의 현실적 모델을 구축합니다:\n\n- **Radiative interior**: rigid rotation at ~430 nHz / 복사 내부: ~430 nHz 강체 자전\n- **Tachocline**: smooth transition at ~0.69 $R_\\odot$ / 타코클라인: ~0.69 $R_\\odot$에서 부드러운 전이\n- **Convection zone**: latitude-dependent differential rotation / 대류층: 위도 의존 차등 자전\n- **Near-surface shear**: slight decrease above 0.95 $R_\\odot$ / 표면 근처 전단: 0.95 $R_\\odot$ 위에서 약간 감소

In [ ]:
def solar_rotation_model(r, theta, r_tach=0.69, gamma_tach=0.02):\n    \"\"\"Model the solar internal rotation rate Omega/(2*pi) in nHz.\n\n    Based on observations summarized in Howe (2009):\n    - Rigid rotation in radiative interior at ~430 nHz\n    - Tachocline transition at r_tach with width gamma_tach\n    - Surface differential rotation: Eq. (25)/(26) in the paper\n    - Near-surface shear above 0.95 R_sun\n\n    Args:\n        r: Fractional radius r/R_sun (scalar or array).\n        theta: Colatitude in radians (0=pole, pi/2=equator).\n        r_tach: Tachocline center in R_sun.\n        gamma_tach: Tachocline half-width in R_sun.\n\n    Returns:\n        Rotation rate Omega/(2*pi) in nHz.\n    \"\"\"\n    r = np.asarray(r, dtype=float)\n    theta = np.asarray(theta, dtype=float)\n\n    # Rigid rotation rate of the radiative interior\n    omega_core = 430.0  # nHz\n\n    # Surface differential rotation (Eq. 26 — Doppler/plasma)\n    mu = np.sin(np.pi / 2 - theta)  # sin(latitude) = cos(colatitude)\n    omega_surface = 452.0 - 49.0 * mu**2 - 84.0 * mu**4  # nHz\n\n    # Tachocline transition (smooth step function)\n    tach_transition = 0.5 * (1.0 + np.tanh((r - r_tach) / gamma_tach))\n\n    # Rotation in convection zone (interpolated)\n    omega_cz = omega_core + tach_transition * (omega_surface - omega_core)\n\n    # Near-surface shear: rotation increases slightly below 0.95 R_sun\n    # then decreases to the surface value\n    r_shear = 0.95\n    shear_amplitude = 10.0  # nHz — peak excess rotation below surface\n    shear_mask = np.where(r > r_shear, 1.0, 0.0)\n    shear_profile = shear_amplitude * np.exp(-((r - r_shear) / 0.02)**2)\n    omega_cz = omega_cz + shear_profile * tach_transition\n\n    return omega_cz\n\n\n# Create grid\nr_grid = np.linspace(0.0, 1.0, 500)\ntheta_grid = np.linspace(0.001, np.pi - 0.001, 400)  # colatitude\nR, THETA = np.meshgrid(r_grid, theta_grid)\n\n# Compute rotation rate on grid\nOMEGA = solar_rotation_model(R, THETA)\n\nprint(f\"Rotation rate range: {OMEGA.min():.1f} - {OMEGA.max():.1f} nHz\")\nprint(f\"Equatorial surface: {solar_rotation_model(0.99, np.pi/2):.1f} nHz\")\nprint(f\"Polar surface: {solar_rotation_model(0.99, 0.01):.1f} nHz\")\nprint(f\"Core: {solar_rotation_model(0.2, np.pi/2):.1f} nHz\")

### Cross-section plot / 단면도 (cf. Figure 1 in the paper)\n\nVisualize the rotation profile as a meridional cross-section, similar to Figure 1 of the review.\n리뷰의 Figure 1과 유사한 자오면 단면도로 자전 프로파일을 시각화합니다.

In [ ]:
# Convert to Cartesian for cross-section plot\nX = R * np.sin(THETA)\nY = R * np.cos(THETA)\n\nfig, ax = plt.subplots(figsize=(8, 8))\n\n# Plot rotation rate as filled contours\nlevels = np.linspace(280, 470, 20)\ncf = ax.contourf(X, Y, OMEGA, levels=levels, cmap=\"jet\", extend=\"both\")\nax.contour(X, Y, OMEGA, levels=levels, colors=\"k\", linewidths=0.3, alpha=0.5)\n\n# Mark key boundaries\ntheta_circle = np.linspace(0, 2 * np.pi, 200)\nfor r_boundary, label, ls in [\n    (0.69, \"Tachocline\", \"--\"),\n    (0.713, \"CZ base\", \":\"),\n    (0.95, \"Near-surface shear\", \"-.\"),\n    (1.0, \"Surface\", \"-\"),\n]:\n    ax.plot(\n        r_boundary * np.cos(theta_circle),\n        r_boundary * np.sin(theta_circle),\n        ls,\n        color=\"white\" if r_boundary < 1.0 else \"black\",\n        linewidth=1.0,\n        alpha=0.7,\n    )\n\n# Annotations\nax.annotate(\"Core\", xy=(0, 0.15), fontsize=9, ha=\"center\", color=\"white\")\nax.annotate(\"Radiative\\nInterior\", xy=(0, 0.45), fontsize=9, ha=\"center\", color=\"white\")\nax.annotate(\"Tachocline\", xy=(0.55, 0.45), fontsize=8, ha=\"center\", color=\"white\", rotation=60)\nax.annotate(\"Convection\\nZone\", xy=(0, 0.82), fontsize=9, ha=\"center\", color=\"black\")\nax.annotate(\"Near-surface\\nshear\", xy=(0.65, 0.72), fontsize=8, ha=\"center\", color=\"black\")\n\ncbar = plt.colorbar(cf, ax=ax, shrink=0.7, label=r\"$\\Omega/2\\pi$ (nHz)\")\n\nax.set_xlim(-1.1, 1.1)\nax.set_ylim(-1.1, 1.1)\nax.set_aspect(\"equal\")\nax.set_xlabel(r\"$x / R_\\odot$\")\nax.set_ylabel(r\"$y / R_\\odot$\")\nax.set_title(\"Solar Interior Rotation Profile (Model)\\n태양 내부 자전 프로파일 (모델)\")\nplt.tight_layout()\nplt.show()

### Radial cuts at constant latitude / 일정 위도에서의 반경 방향 절단면 (cf. Figure 19 right panel)\n\nPlot rotation rate as a function of radius at several latitudes, as in Figure 19 (right panel) of the review.\n리뷰 Figure 19 (오른쪽 패널)와 같이 여러 위도에서 반경의 함수로 자전율을 도시합니다.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))\n\nr_plot = np.linspace(0.4, 1.0, 500)\nlatitudes = [0, 15, 30, 45, 60, 75]  # degrees\ncolors = plt.cm.coolwarm(np.linspace(0.95, 0.05, len(latitudes)))\n\nfor lat, c in zip(latitudes, colors):\n    theta = np.pi / 2 - np.radians(lat)  # colatitude\n    omega = solar_rotation_model(r_plot, theta)\n    ax.plot(r_plot, omega, color=c, linewidth=2, label=f\"{lat}°\")\n\n# Mark key radii\nfor r_mark, label in [(0.69, \"Tachocline\"), (0.713, \"CZ base\"), (0.95, \"Shear\")]:\n    ax.axvline(r_mark, color=\"gray\", linestyle=\"--\", alpha=0.5, linewidth=0.8)\n    ax.text(r_mark + 0.005, 290, label, fontsize=7, rotation=90, va=\"bottom\", color=\"gray\")\n\nax.set_xlabel(r\"$r / R_\\odot$\")\nax.set_ylabel(r\"$\\Omega / 2\\pi$ (nHz)\")\nax.set_title(\"Rotation Rate vs. Radius at Constant Latitude\\n일정 위도에서의 자전율 대 반경\")\nax.legend(title=\"Latitude\", loc=\"lower left\")\nax.set_xlim(0.4, 1.02)\nax.set_ylim(280, 470)\nax.grid(True, alpha=0.3)\nplt.tight_layout()\nplt.show()

## 2. Slanted Contours vs. Cylindrical vs. Radial / 기울어진 윤곽 vs. 원통형 vs. 반경형\n\nOne of the key findings (Section 7.4) is that constant-rotation contours are tilted at ~25° to the rotation axis — neither cylindrical (0°, Taylor-Proudman) nor radial (90°). We compare these three configurations.\n\n핵심 발견(Section 7.4): 등자전 윤곽이 자전축에 대해 ~25° 기울어짐 — 원통형(0°)도 반경형(90°)도 아님. 세 구성을 비교합니다. (cf. Figure 20)

In [ ]:
def make_rotation_profile(r, theta, mode=\"slanted\"):\n    \"\"\"Generate idealized rotation profiles for comparison.\n\n    Args:\n        r: Fractional radius array.\n        theta: Colatitude in radians.\n        mode: 'cylindrical', 'radial', or 'slanted' (25 deg).\n\n    Returns:\n        Rotation rate in nHz.\n    \"\"\"\n    # Surface rotation at 0.99 R_sun matched to observations\n    mu_surface = np.cos(theta)\n    omega_surface = 452.0 - 49.0 * mu_surface**2 - 84.0 * mu_surface**4\n    omega_core = 430.0\n\n    # Tachocline transition\n    tach = 0.5 * (1.0 + np.tanh((r - 0.69) / 0.02))\n\n    if mode == \"cylindrical\":\n        # Rotation depends on distance from rotation axis: s = r * sin(theta)\n        s = r * np.sin(theta)\n        # Map: s=0 (pole) -> omega_pole, s=1 (equator) -> omega_equator\n        s_clipped = np.clip(s, 0, 1)\n        mu_eff = np.sqrt(1 - s_clipped**2)\n        omega_cz = 452.0 - 49.0 * mu_eff**2 - 84.0 * mu_eff**4\n    elif mode == \"radial\":\n        # Rotation depends only on colatitude (constant along radial lines)\n        omega_cz = omega_surface\n    elif mode == \"slanted\":\n        # Rotation constant along lines tilted 25 deg from rotation axis\n        angle = np.radians(25)\n        # Effective coordinate along the slanted direction\n        s_eff = r * np.sin(theta + angle * (1 - r))  # approximate\n        s_eff = np.clip(s_eff, 0, 1)\n        mu_eff = np.sqrt(np.clip(1 - s_eff**2, 0, 1))\n        omega_cz = 452.0 - 49.0 * mu_eff**2 - 84.0 * mu_eff**4\n    else:\n        raise ValueError(f\"Unknown mode: {mode}\")\n\n    return omega_core + tach * (omega_cz - omega_core)\n\n\n# Grid for upper-right quadrant only\nr_q = np.linspace(0.5, 1.0, 300)\ntheta_q = np.linspace(0.001, np.pi / 2, 300)\nRq, THq = np.meshgrid(r_q, theta_q)\nXq = Rq * np.sin(THq)  # horizontal\nYq = Rq * np.cos(THq)  # vertical (rotation axis)\n\nfig, axes = plt.subplots(1, 3, figsize=(15, 5))\nmodes = [\"cylindrical\", \"radial\", \"slanted\"]\ntitles = [\n    \"Cylindrical\\n원통형 (Taylor-Proudman)\",\n    \"Radial\\n반경형\",\n    \"Slanted (~25°)\\n기울어진 (~25°, observed)\",\n]\nlevels = np.linspace(300, 465, 18)\n\nfor ax, mode, title in zip(axes, modes, titles):\n    omega_q = make_rotation_profile(Rq, THq, mode=mode)\n    cf = ax.contourf(Xq, Yq, omega_q, levels=levels, cmap=\"jet\")\n    ax.contour(Xq, Yq, omega_q, levels=levels, colors=\"k\", linewidths=0.3)\n\n    # Draw boundaries\n    th_arc = np.linspace(0, np.pi / 2, 100)\n    ax.plot(np.sin(th_arc), np.cos(th_arc), \"k-\", linewidth=1.5)\n    ax.plot(0.69 * np.sin(th_arc), 0.69 * np.cos(th_arc), \"w--\", linewidth=0.8)\n    ax.plot([0, 1], [0, 0], \"k-\", linewidth=0.5)\n    ax.plot([0, 0], [0, 1], \"k-\", linewidth=0.5)\n\n    ax.set_xlim(0, 1.05)\n    ax.set_ylim(0, 1.05)\n    ax.set_aspect(\"equal\")\n    ax.set_title(title)\n    ax.set_xlabel(r\"$r \\sin\\theta / R_\\odot$\")\n\naxes[0].set_ylabel(r\"$r \\cos\\theta / R_\\odot$\")\nplt.colorbar(cf, ax=axes, shrink=0.8, label=r\"$\\Omega/2\\pi$ (nHz)\")\nplt.suptitle(\n    \"Comparison of Rotation Configurations (cf. Howe 2009, Fig. 20)\\n\"\n    \"자전 배치 비교 (Howe 2009, Fig. 20 참조)\",\n    y=1.02,\n)\nplt.tight_layout()\nplt.show()

## 3. Simplified 1D Rotation Inversion (RLS) / 간소화된 1D 자전 역산 (RLS)\n\nWe implement a simplified 1D rotation inversion following the RLS approach (Section 3.4, Eq. 21). In the 1D case, we have:\n1D 역산을 RLS 접근법(Section 3.4, Eq. 21)에 따라 간소화하여 구현합니다:\n\n$$d_i = \\int_0^{R_\\odot} K_i(r) \\Omega(r) \\, dr + \\epsilon_i$$\n\nWe generate synthetic data from our rotation model, then invert to recover the profile.\n자전 모델에서 합성 데이터를 생성한 후 역산으로 프로파일을 복원합니다.

In [ ]:
def make_gaussian_kernels(r_grid, turning_points, kernel_width=0.05):\n    \"\"\"Create simplified Gaussian rotation kernels for 1D inversion.\n\n    In real helioseismology, kernels are computed from mode eigenfunctions.\n    Here we use Gaussians centered near the lower turning point, representing\n    the approximate sensitivity of each mode.\n\n    Args:\n        r_grid: Radial grid points.\n        turning_points: Lower turning point radii for each mode.\n        kernel_width: Width of Gaussian kernels.\n\n    Returns:\n        Kernel matrix K[i, j] where i=mode, j=radial grid point.\n    \"\"\"\n    n_modes = len(turning_points)\n    n_r = len(r_grid)\n    K = np.zeros((n_modes, n_r))\n    dr = r_grid[1] - r_grid[0]\n\n    for i, r_t in enumerate(turning_points):\n        # Kernel peaks near turning point, extends to surface\n        # Amplitude weighted by r^2 (volume element)\n        for j, r in enumerate(r_grid):\n            if r >= r_t:\n                # Simplified kernel: peaks at turning point, decays toward surface\n                K[i, j] = np.exp(-((r - r_t) / kernel_width) ** 2) * r**2\n                # Add a broader component representing sensitivity in upper layers\n                K[i, j] += 0.3 * r**2 * np.exp(-((r - 0.9) / 0.2) ** 2)\n\n        # Normalize\n        norm = np.sum(K[i, :]) * dr\n        if norm > 0:\n            K[i, :] /= norm\n\n    return K\n\n\ndef rls_inversion_1d(data, kernels, r_grid, sigma, mu=1e-3):\n    \"\"\"Perform 1D RLS rotation inversion.\n\n    Minimizes: sum_i [(d_i - integral K_i * Omega dr)^2 / sigma_i^2]\n              + mu * integral (d^2 Omega / dr^2)^2 dr\n\n    This is the 1D version of Eq. (21) in the paper.\n\n    Args:\n        data: Observed splitting data (n_modes,).\n        kernels: Kernel matrix (n_modes, n_r).\n        r_grid: Radial grid.\n        sigma: Uncertainties on data.\n        mu: Regularization parameter (smoothness tradeoff).\n\n    Returns:\n        Inferred rotation profile on r_grid.\n    \"\"\"\n    n_modes, n_r = kernels.shape\n    dr = r_grid[1] - r_grid[0]\n\n    # Build the normal equations: (K^T W K + mu H) Omega = K^T W d\n    # W = diag(1/sigma^2), H = second-derivative smoothing matrix\n    W = np.diag(1.0 / sigma**2)\n    KtWK = kernels.T @ W @ kernels * dr**2\n    KtWd = kernels.T @ W @ data * dr\n\n    # Second-derivative smoothing matrix\n    H = np.zeros((n_r, n_r))\n    for i in range(1, n_r - 1):\n        H[i, i - 1] = 1.0\n        H[i, i] = -2.0\n        H[i, i + 1] = 1.0\n    H /= dr**2\n    HtH = H.T @ H * dr\n\n    # Solve\n    A = KtWK + mu * HtH\n    omega_inferred = np.linalg.solve(A, KtWd)\n\n    return omega_inferred\n\n\n# Set up the inversion experiment\nnp.random.seed(42)\nn_r = 200\nr_inv = np.linspace(0.05, 0.99, n_r)\n\n# True equatorial rotation profile\ntheta_eq = np.pi / 2  # equator\nomega_true = solar_rotation_model(r_inv, theta_eq)\n\n# Simulate modes with various turning points (representing different l values)\n# Lower l -> deeper turning point\nturning_points = np.concatenate([\n    np.linspace(0.1, 0.4, 5),     # low-l modes (l=1-5) -> deep\n    np.linspace(0.4, 0.7, 10),    # medium-low l\n    np.linspace(0.7, 0.85, 15),   # medium l\n    np.linspace(0.85, 0.95, 20),  # high l\n])\nn_modes = len(turning_points)\n\n# Create kernels\nK = make_gaussian_kernels(r_inv, turning_points, kernel_width=0.04)\n\n# Generate synthetic data (forward problem: d_i = integral K_i * Omega dr)\ndr = r_inv[1] - r_inv[0]\ndata_clean = K @ omega_true * dr\n\n# Add noise\nnoise_level = 2.0  # nHz\nsigma = np.full(n_modes, noise_level)\ndata_noisy = data_clean + np.random.randn(n_modes) * noise_level\n\nprint(f\"Number of modes: {n_modes}\")\nprint(f\"Turning point range: {turning_points.min():.2f} - {turning_points.max():.2f} R_sun\")\nprint(f\"Data range: {data_noisy.min():.1f} - {data_noisy.max():.1f} nHz\")

In [ ]:
# Perform inversions with different regularization parameters\n# to demonstrate the resolution-error tradeoff\nmu_values = [1e-5, 1e-3, 1e-1]\nlabels = [r\"$\\mu = 10^{-5}$ (under-regularized)\",\n          r\"$\\mu = 10^{-3}$ (optimal)\",\n          r\"$\\mu = 10^{-1}$ (over-regularized)\"]\ncolors_inv = [\"tab:orange\", \"tab:blue\", \"tab:green\"]\n\nfig, axes = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={\"height_ratios\": [3, 1]})\n\n# Top panel: inversion results\nax = axes[0]\nax.plot(r_inv, omega_true, \"k-\", linewidth=2, label=\"True profile / 실제 프로파일\", zorder=5)\n\nfor mu, label, c in zip(mu_values, labels, colors_inv):\n    omega_inv = rls_inversion_1d(data_noisy, K, r_inv, sigma, mu=mu)\n    ax.plot(r_inv, omega_inv, color=c, linewidth=1.5, alpha=0.8, label=label)\n\nax.axvline(0.69, color=\"gray\", linestyle=\"--\", alpha=0.5)\nax.text(0.695, 425, \"Tachocline\", fontsize=8, color=\"gray\")\nax.set_xlim(0.1, 1.0)\nax.set_ylim(410, 470)\nax.set_ylabel(r\"$\\Omega / 2\\pi$ (nHz)\")\nax.set_title(\n    \"1D RLS Inversion: Regularization Tradeoff (Equatorial, cf. Sec. 3.4)\\n\"\n    \"1D RLS 역산: 정규화 균형 (적도, Section 3.4 참조)\"\n)\nax.legend(loc=\"lower right\", fontsize=9)\nax.grid(True, alpha=0.3)\n\n# Bottom panel: sample kernels\nax2 = axes[1]\nfor i in [0, 10, 25, 40]:\n    ax2.plot(r_inv, K[i, :], linewidth=1.2,\n             label=f\"$r_t$ = {turning_points[i]:.2f} $R_\\odot$\")\nax2.set_xlabel(r\"$r / R_\\odot$\")\nax2.set_ylabel(\"Kernel amplitude\")\nax2.set_title(\"Sample Rotation Kernels / 자전 커널 예시\")\nax2.legend(fontsize=8, ncol=2)\nax2.set_xlim(0.1, 1.0)\nax2.grid(True, alpha=0.3)\n\nplt.tight_layout()\nplt.show()

## 4. Torsional Oscillation Pattern / 비틀림 진동 패턴\n\nThe torsional oscillation (Section 9) is a pattern of migrating bands of faster/slower-than-average zonal flow. We simulate this pattern as a function of latitude and time, based on the observational description.\n\n비틀림 진동(Section 9)은 평균보다 빠르거나 느린 대상 흐름 대의 이동 패턴입니다. 관측 기술을 바탕으로 위도와 시간의 함수로 시뮬레이션합니다. (cf. Figures 24, 30)

In [ ]:
def torsional_oscillation(lat, t, period=11.0):\n    \"\"\"Model the torsional oscillation pattern.\n\n    Based on the observational description in Section 9:\n    - Equatorward-migrating bands at low/mid latitudes\n    - Poleward-migrating bands at high latitudes\n    - Amplitude ~1-2 nHz (a few m/s)\n\n    Args:\n        lat: Latitude in degrees (-90 to 90).\n        t: Time in years.\n        period: Solar cycle period in years.\n\n    Returns:\n        Rotation rate residual delta_Omega/(2*pi) in nHz.\n    \"\"\"\n    lat = np.asarray(lat, dtype=float)\n    t = np.asarray(t, dtype=float)\n\n    omega_freq = 2 * np.pi / period\n\n    # Equatorward branch: starts at ~40 deg, migrates to equator over ~11 years\n    # Phase increases with latitude (higher latitude = earlier phase)\n    lat_abs = np.abs(lat)\n    equatorward_phase = omega_freq * t + np.radians(lat_abs) * 3.0\n    equatorward = 1.5 * np.sin(equatorward_phase)\n    # Taper at high latitudes\n    equatorward *= np.exp(-((lat_abs - 15) / 20) ** 2)\n    equatorward *= (lat_abs < 50)\n\n    # Poleward branch: starts at ~40-50 deg, migrates to pole\n    poleward_phase = omega_freq * t - np.radians(lat_abs - 50) * 2.5\n    poleward = 1.0 * np.sin(poleward_phase)\n    poleward *= np.exp(-((lat_abs - 60) / 15) ** 2)\n    poleward *= (lat_abs > 35)\n\n    return equatorward + poleward\n\n\n# Create latitude-time diagram\nlat_to = np.linspace(-70, 70, 300)\nt_to = np.linspace(1986, 2009, 500)\nLAT_TO, T_TO = np.meshgrid(lat_to, t_to)\n\nDELTA_OMEGA = torsional_oscillation(LAT_TO, T_TO)\n\nfig, axes = plt.subplots(2, 1, figsize=(12, 8))\n\n# Top: torsional oscillation pattern\nax = axes[0]\ncf = ax.contourf(\n    T_TO, LAT_TO, DELTA_OMEGA,\n    levels=np.linspace(-2, 2, 21),\n    cmap=\"RdBu_r\",\n    extend=\"both\",\n)\nax.set_xlabel(\"Year / 연도\")\nax.set_ylabel(\"Latitude / 위도 (°)\")\nax.set_title(\n    \"Simulated Torsional Oscillation Pattern (cf. Howe 2009, Figs. 24, 30)\\n\"\n    \"비틀림 진동 패턴 시뮬레이션\"\n)\ncbar = plt.colorbar(cf, ax=ax, label=r\"$\\delta\\Omega / 2\\pi$ (nHz)\")\nax.axhline(0, color=\"k\", linewidth=0.5, alpha=0.3)\n\n# Add approximate solar cycle phase markers\nfor year_max in [1989.5, 2000.5]:\n    ax.axvline(year_max, color=\"gray\", linestyle=\"--\", alpha=0.5)\n    ax.text(year_max, 65, \"Max\", fontsize=8, ha=\"center\", color=\"gray\")\nfor year_min in [1986, 1996, 2008]:\n    ax.axvline(year_min, color=\"gray\", linestyle=\":\", alpha=0.5)\n    ax.text(year_min, 65, \"Min\", fontsize=8, ha=\"center\", color=\"gray\")\n\n# Bottom: time series at selected latitudes\nax2 = axes[1]\nt_series = np.linspace(1986, 2009, 500)\nfor lat_sel, c in zip([0, 15, 30, 45, 60], plt.cm.viridis(np.linspace(0, 0.9, 5))):\n    delta = torsional_oscillation(lat_sel, t_series)\n    ax2.plot(t_series, delta, color=c, linewidth=1.5, label=f\"{lat_sel}°\")\nax2.set_xlabel(\"Year / 연도\")\nax2.set_ylabel(r\"$\\delta\\Omega / 2\\pi$ (nHz)\")\nax2.set_title(\"Torsional Oscillation at Selected Latitudes / 선택된 위도에서의 비틀림 진동\")\nax2.legend(title=\"Latitude\", ncol=5, fontsize=9)\nax2.axhline(0, color=\"k\", linewidth=0.5, alpha=0.3)\nax2.grid(True, alpha=0.3)\n\nplt.tight_layout()\nplt.show()

## 5. Near-Surface Shear: Magnetic vs. Doppler Rotation / 표면 근처 전단: 자기 추적자 vs. Doppler\n\nEquations (25) and (26) give different rotation rates for magnetic features and surface plasma, evidence for the near-surface shear layer (Section 8).\n\n식 (25)와 (26)은 자기 요소와 표면 플라즈마에 대해 다른 자전율을 주며, 이는 표면 근처 전단층의 증거입니다.

In [ ]:
lat_range = np.linspace(0, 75, 200)\nmu = np.sin(np.radians(lat_range))\n\n# Eq. 25: magnetic features (sunspot tracking)\nomega_mag = 462 - 74 * mu**2 - 53 * mu**4  # nHz\n\n# Eq. 26: surface plasma (Doppler)\nomega_dop = 452 - 49 * mu**2 - 84 * mu**4  # nHz\n\n# Convert to sidereal period in days\nperiod_mag = 1e9 / (omega_mag * 86400)  # days\nperiod_dop = 1e9 / (omega_dop * 86400)  # days\n\nfig, axes = plt.subplots(1, 2, figsize=(14, 5))\n\n# Left: rotation rate\nax = axes[0]\nax.plot(lat_range, omega_mag, \"r-\", linewidth=2,\n        label=r\"Magnetic features: $462 - 74\\mu^2 - 53\\mu^4$ (Eq. 25)\")\nax.plot(lat_range, omega_dop, \"b-\", linewidth=2,\n        label=r\"Doppler plasma: $452 - 49\\mu^2 - 84\\mu^4$ (Eq. 26)\")\nax.fill_between(lat_range, omega_dop, omega_mag, alpha=0.15, color=\"purple\",\n                label=\"Near-surface shear / 표면 근처 전단\")\nax.set_xlabel(\"Latitude / 위도 (°)\")\nax.set_ylabel(r\"$\\Omega / 2\\pi$ (nHz)\")\nax.set_title(\"Surface Differential Rotation: Two Tracers\\n표면 차등 자전: 두 추적자\")\nax.legend(fontsize=8)\nax.grid(True, alpha=0.3)\nax.set_xlim(0, 75)\n\n# Right: difference (shear signature)\nax2 = axes[1]\ndiff = omega_mag - omega_dop\nax2.plot(lat_range, diff, \"purple\", linewidth=2)\nax2.fill_between(lat_range, 0, diff, alpha=0.2, color=\"purple\")\nax2.set_xlabel(\"Latitude / 위도 (°)\")\nax2.set_ylabel(r\"$\\Delta\\Omega / 2\\pi$ (nHz)\")\nax2.set_title(\n    \"Rotation Rate Difference (Magnetic - Doppler)\\n\"\n    \"자전율 차이 (자기 추적자 - Doppler)\"\n)\nax2.annotate(\n    \"Magnetic tracers rotate faster\\nbecause they are anchored\\nin the faster-rotating subsurface layer\\n\"\n    \"자기 추적자가 더 빠르게 회전:\\n더 빠른 아래층에 고정되어 있기 때문\",\n    xy=(20, diff[int(20/75*200)]),\n    xytext=(35, 15),\n    fontsize=8,\n    arrowprops=dict(arrowstyle=\"->\", color=\"gray\"),\n    bbox=dict(boxstyle=\"round\", fc=\"lightyellow\", alpha=0.8),\n)\nax2.set_xlim(0, 75)\nax2.grid(True, alpha=0.3)\n\nplt.tight_layout()\nplt.show()\n\nprint(f\"\\nEquatorial difference: {462-452:.0f} nHz = {(462-452)/452*100:.1f}%\")\nprint(f\"At 30° latitude: {diff[int(30/75*200)]:.1f} nHz\")\nprint(f\"At 60° latitude: {diff[int(60/75*200)]:.1f} nHz\")

## Summary / 요약\n\nThis notebook implemented key concepts from Howe (2009):\n이 노트북은 Howe (2009)의 핵심 개념을 구현했습니다:\n\n1. **Solar rotation model** — A parametric model capturing all four major rotation features (rigid interior, tachocline, differential convection zone, near-surface shear)\n   태양 자전 모델 — 4대 자전 구조를 포착하는 매개변수 모델\n\n2. **Slanted contour comparison** — Visualization of cylindrical, radial, and 25°-slanted rotation configurations, demonstrating why the observed contours are a critical constraint\n   기울어진 등자전 윤곽 비교 — 원통형, 반경형, 25° 기울어진 구성의 시각화\n\n3. **1D RLS inversion** — Simplified demonstration of the regularized least-squares approach with resolution-error tradeoff\n   1D RLS 역산 — 정규화 최소제곱법의 해상도-오차 균형 시연\n\n4. **Torsional oscillation** — Latitude-time diagram of migrating zonal flow bands\n   비틀림 진동 — 이동하는 대상 흐름 대의 위도-시간 다이어그램\n\n5. **Near-surface shear** — Comparison of magnetic tracer and Doppler rotation rates (Eqs. 25-26)\n   표면 근처 전단 — 자기 추적자와 Doppler 자전율 비교